In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import sys

sys.path.append('..')

from sklearn import set_config
set_config(transform_output="pandas")

from sklearn.neural_network import MLPRegressor
from scipy.stats import randint, uniform, loguniform

from src.utils import DEV_SET_CLEAN_NORMAL_PATH, COLS
from src.pipeline import build_feature_pipeline, build_target_pipeline
from src.model_evaluation import evaluate_models_cv, grid_search_cv
from src.plots import plot_cv_results, plot_grid_search_results

OVERFITTING_CUTOFF = 70

# ===========================
# CARGA DEL DATASET NORMAL
# ===========================
df_normal = pd.read_csv("../" + DEV_SET_CLEAN_NORMAL_PATH)
X_normal = df_normal.drop(columns=[COLS.TARGET])
y_normal = df_normal[COLS.TARGET]

# ===========================
# PIPELINES
# ===========================
from src.config import NN_CONFIG_NORMAL  # si querés un config propio
# Si no definiste NN_CONFIG_NORMAL, podés usar TREE_BASED_CONFIG_NORMAL
feature_pipeline_normal = build_feature_pipeline(NN_CONFIG_NORMAL)
target_pipeline_normal = build_target_pipeline(NN_CONFIG_NORMAL)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ModuleNotFoundError: No module named 'tqdm'

In [ ]:
base_nn_models = {
    "mlp_relu": MLPRegressor(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        learning_rate="adaptive",
        max_iter=400,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42
    )
}

base_nn_results = evaluate_models_cv(
    base_nn_models,
    X_normal,
    y_normal,
    feature_pipeline_normal,
    target_pipeline_normal
)

base_nn_results


In [ ]:
mlp_random_search_params = {
    "hidden_layer_sizes": [
        (64,), (128,), (256,),
        (64, 32), (128, 64), (256, 128),
        (128, 128, 64)
    ],
    "alpha": loguniform(1e-6, 1e-2),
    "learning_rate_init": loguniform(1e-4, 1e-2),
    "batch_size": [64, 128, 256],

    "activation": ["relu"],
    "solver": ["adam"],
    "learning_rate": ["adaptive"],
    "max_iter": [400],
    "early_stopping": [True],
    "validation_fraction": [0.1],
    "random_state": [42],
}

mlp_grid_results = grid_search_cv(
    model_class=MLPRegressor,
    param_grid=mlp_random_search_params,
    X=X_normal,
    y=y_normal,
    feature_pipeline=feature_pipeline_normal,
    target_pipeline=target_pipeline_normal,
    n_splits=3,
    n_iter=120,   # ajustalo para GPU/tiempo
    verbose=True
)

mlp_grid_results.to_csv("../results/random_search/mlp_normal.csv")


ValueError: Specifying the columns using strings is only supported for dataframes.

In [ ]:
def get_best_not_overfitted(results):
    results_out = results.copy()
    return results_out[results_out['overfit_gap_%'] < OVERFITTING_CUTOFF].sort_values('rmse_mean')

mlp_grid_results_cut = get_best_not_overfitted(mlp_grid_results)
mlp_grid_results_cut.head()


In [ ]:
plot_grid_search_results(mlp_grid_results, metric="rmse_mean")
plot_grid_search_results(mlp_grid_results_cut, metric="rmse_mean")
